# 106 — Reverse Delta-ML (Test Compounds as Templates)

**Motivation:** Standard delta-ML uses training compounds as templates for test queries. But the test set is an analog expansion of training — many test compounds are *closer to each other* than to any single training compound. We can exploit the test set's internal structure transductively.

**Strategy — Two-stage transductive prediction:**

Stage 1 — Standard multi-template delta (from nb97): get initial test predictions `te_pred_v1`

Stage 2 — Reverse delta refinement:
- For each training compound in validation fold, find its nearest test neighbor(s)
- If the test neighbor is close (sim > SIM_REFINE), use `te_pred_v1` as a pseudo-label for that test compound
- Apply reverse delta: `val_refined = te_pred_v1[best_te] - delta(te->val)` (using the same delta LGBM)
- Blend: `final = (1 - sim_weight) * direct + sim_weight * val_refined`
  where `sim_weight = clip(best_te_sim - SIM_REFINE, 0, 1) * REFINE_STRENGTH`

**Why this helps:** Test compounds' initial predictions (from training templates) are already good. When a validation compound is more similar to a test compound than to any training compound, routing through the test compound's prediction adds signal.

**Caution:** This creates a circular dependency: val predictions depend on test predictions which were computed using training data. We break this carefully: the delta model is trained on training pairs only; test predictions use only training templates.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
SIM_LO = 0.35; SIM_HI = 0.90   # standard delta window
SIM_REFINE = 0.45               # min similarity to use test neighbor for refinement
K_NEIGHBORS = 10

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)
phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Computing physchem...


Train 4,139  Test 513  Cliffs 0


In [4]:
# --- Precompute test-train and train-train similarity ---
print("Computing train-train Tanimoto...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum_tr = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum_tr[:,None] + rowsum_tr[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

print("Computing test-train Tanimoto...", flush=True)
dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)

print("Computing train-test Tanimoto...", flush=True)
sim_tr_te = sim_te_tr.T  # (N_train, N_test)

print(f"Test-train sim: mean={sim_te_tr.max(1).mean():.3f}  "
      f"best test-train sim median={np.median(sim_te_tr.max(1)):.3f}")

Computing train-train Tanimoto...


Computing test-train Tanimoto...


Computing train-test Tanimoto...


Test-train sim: mean=0.532  best test-train sim median=0.523


In [5]:
# --- Build delta model (same as nb97) ---
MAX_PAIRS = 400_000
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common); d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window: {len(i_idx):,}")
rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
F_ij = make_delta_feats(fps_tr[i_idx], fps_tr[j_idx], sims_ij, y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_tr[j_idx], fps_tr[i_idx], sims_ij, y_tr[j_idx], -phys_diff_ij)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_tr[j_idx]-y_tr[i_idx], y_tr[i_idx]-y_tr[j_idx]])
print(f"Delta dataset: {F_all.shape}  delta range [{y_all.min():.2f}, {y_all.max():.2f}]")

print("Training delta LGBM...", flush=True)
DELTA_LGBM = dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
delta_model = lgb.LGBMRegressor(**DELTA_LGBM)
delta_model.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("Delta model trained.", flush=True)

Pairs in sim window: 5,177


Delta dataset: (10354, 137)  delta range [-4.68, 4.68]
Training delta LGBM...


Delta model trained.


In [6]:
# --- Standard multi-template delta (Stage 1) ---
def multi_template_predict(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                            sim_matrix, delta_model, direct_preds,
                            sim_lo=SIM_LO, sim_hi=SIM_HI, k=K_NEIGHBORS):
    """Returns (preds, n_templates)."""
    N = len(fps_query)
    preds = np.full(N, np.nan)
    n_templates = np.zeros(N, dtype=int)
    for qi in range(N):
        sim_row = sim_matrix[qi]
        cand_mask = (sim_row >= sim_lo) & (sim_row <= sim_hi)
        cand_idx = np.where(cand_mask)[0]
        if len(cand_idx) == 0:
            preds[qi] = direct_preds[qi]; continue
        top_k = np.argsort(-sim_row[cand_idx])[:k]
        cand_idx = cand_idx[top_k]; cand_sims = sim_row[cand_idx]
        n_templates[qi] = len(cand_idx)
        fp_q_rep = np.tile(fps_query[qi:qi+1], (len(cand_idx), 1))
        fp_refs  = fps_ref[cand_idx]
        sims_col = cand_sims[:,None]; anc_pec50 = y_ref[cand_idx]
        phys_d = phys_query[qi:qi+1] - phys_ref[cand_idx]
        F_k = make_delta_feats(fp_refs, fp_q_rep, sims_col, anc_pec50, phys_d)
        delta_k = delta_model.predict(F_k)
        template_preds = y_ref[cand_idx] + delta_k
        weights = cand_sims ** 2
        preds[qi] = np.average(template_preds, weights=weights)
    return preds, n_templates

print("Standard multi-template predict function ready.")

Standard multi-template predict function ready.


In [7]:
# --- Stage 2: Reverse delta refinement ---
# For each training/val compound, find nearest test neighbor
# If close enough, refine using: val_refined = te_pred[best_te] - delta(te->val)

def reverse_delta_refine(fps_val, fps_te, phys_val, phys_te,
                          sim_val_te, te_preds_v1, direct_val,
                          delta_model, sim_refine=SIM_REFINE, refine_strength=1.0):
    """
    For each val compound, find best test neighbor.
    If best_te_sim >= sim_refine, refine prediction using reverse delta.
    Returns (refined_preds, refine_sim, refine_weights)
    """
    N = len(fps_val)
    refined = np.copy(direct_val).astype(float)
    refine_sim = np.zeros(N)
    refine_w   = np.zeros(N)

    for vi in range(N):
        best_te_idx = sim_val_te[vi].argmax()
        best_te_sim = sim_val_te[vi, best_te_idx]
        refine_sim[vi] = float(best_te_sim)
        if best_te_sim < sim_refine:
            continue
        # Reverse delta: predict delta(te -> val)
        # anchor = test compound (known prediction), query = val compound
        fp_te_i = fps_te[best_te_idx:best_te_idx+1]
        fp_va_i = fps_val[vi:vi+1]
        sim_col = np.array([[best_te_sim]])
        te_pec50_pseudo = np.array([te_preds_v1[best_te_idx]])
        phys_d = phys_val[vi:vi+1] - phys_te[best_te_idx:best_te_idx+1]
        F_rev = make_delta_feats(fp_te_i, fp_va_i, sim_col, te_pec50_pseudo, phys_d)
        delta_rev = delta_model.predict(F_rev)[0]
        val_via_te = te_preds_v1[best_te_idx] + delta_rev
        # Blend weight proportional to how far above threshold the sim is
        w = float(np.clip((best_te_sim - sim_refine) / (1.0 - sim_refine + 1e-6), 0.0, 1.0)) * refine_strength
        w = float(np.clip(w, 0.0, 1.0))
        refine_w[vi] = w
        refined[vi] = w * val_via_te + (1.0 - w) * direct_val[vi]

    return refined, refine_sim, refine_w

print(f"Reverse delta refine function ready (sim_refine={SIM_REFINE})")

Reverse delta refine function ready (sim_refine=0.45)


In [8]:
# --- Full scaffold 5-fold CV ---
# Stage 1: multi-template delta on val using fold-train as templates
# Stage 2: reverse delta refine using test-set predictions from stage 1
# Note: test predictions use ALL training data (not fold-restricted) to avoid
# circular leakage in CV — we treat test as a fixed external anchor.

print("\n=== Scaffold 5-fold CV ===", flush=True)

# First: build test predictions using ALL training data
print("Computing test predictions (all-train stage 1)...", flush=True)
m_all_direct = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct_full = m_all_direct.predict(X_te)
te_stage1, _ = multi_template_predict(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, delta_model, te_direct_full
)
print(f"Test stage1: min={te_stage1.min():.2f} med={np.median(te_stage1):.2f} max={te_stage1.max():.2f}")

oof_stage1   = np.full(len(y_tr), np.nan)
oof_stage2   = np.full(len(y_tr), np.nan)
oof_direct   = np.full(len(y_tr), np.nan)
oof_refine_w = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct LGBM
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    # Stage 1: multi-template delta (val x fold-train)
    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)
    stage1_va, _ = multi_template_predict(
        fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
        sim_vf, delta_model, oof_direct[va_idx]
    )
    oof_stage1[va_idx] = stage1_va

    # Stage 2: reverse delta refinement (val x test)
    # Use test's stage1 predictions as pseudo-labels
    sim_va_te = sim_tr_te[va_idx]  # (N_val, N_test)
    stage2_va, rs, rw = reverse_delta_refine(
        fps_va, fps_te, phys_arr[va_idx], phys_te,
        sim_va_te, te_stage1, stage1_va, delta_model
    )
    oof_stage2[va_idx]   = stage2_va
    oof_refine_w[va_idx] = rw

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_s1  = rae(y_tr[va_idx], oof_stage1[va_idx])
    r_s2  = rae(y_tr[va_idx], oof_stage2[va_idx])
    n_refined = int((rw > 0).sum())
    print(f"  fold {fold+1}  direct={r_dir:.4f}  stage1={r_s1:.4f}  "
          f"stage2={r_s2:.4f}  refined={n_refined}/{len(va_idx)}", flush=True)

m_dir  = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_s1   = full_metrics(y_tr, oof_stage1, cliff_pairs, "stage1_delta")
m_s2   = full_metrics(y_tr, oof_stage2, cliff_pairs, "stage2_reverse")
print(f"\nAvg refine weight: {float(np.nanmean(oof_refine_w)):.4f}")
print("\n" + pd.DataFrame([m_dir, m_s1, m_s2],
                           index=["direct","stage1","stage2"]).round(4).to_string())


=== Scaffold 5-fold CV ===


Computing test predictions (all-train stage 1)...


Test stage1: min=2.80 med=4.72 max=5.74


  fold 1  direct=0.4982  stage1=0.2912  stage2=0.2932  refined=20/828


  fold 2  direct=0.5759  stage1=0.3193  stage2=0.3224  refined=28/828


  fold 3  direct=0.6021  stage1=0.3612  stage2=0.3626  refined=18/828


  fold 4  direct=0.5665  stage1=0.3294  stage2=0.3334  refined=25/828


  fold 5  direct=0.6033  stage1=0.3461  stage2=0.3512  refined=30/827


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [stage1_delta] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237
  [stage2_reverse] RAE=0.3297 MAE=0.3000 R2=0.8155 r=0.9049 rho=0.8737 tau=0.7211

Avg refine weight: 0.0056

           RAE     MAE      R2  Pearson  Spearman  Kendall
direct  0.5643  0.5134  0.5991   0.7740    0.7268   0.5345
stage1  0.3266  0.2972  0.8178   0.9060    0.8754   0.7237
stage2  0.3297  0.3000  0.8155   0.9049    0.8737   0.7211


In [9]:
# --- Sweep refine_strength and sim_refine to find best combo ---
print("\nSweeping refine_strength...", flush=True)
best_rs, best_rae_v = 1.0, full_metrics(y_tr, oof_stage2)["RAE"]
for rs_val in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    # Recompute oof_stage2 with this refine_strength
    oof_s2_test = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        sim_va_te = sim_tr_te[va_idx]
        stage2_va, _, _ = reverse_delta_refine(
            fps_tr[va_idx], fps_te, phys_arr[va_idx], phys_te,
            sim_va_te, te_stage1, oof_stage1[va_idx], delta_model,
            refine_strength=rs_val
        )
        oof_s2_test[va_idx] = stage2_va
    mask = np.isfinite(oof_s2_test)
    r = rae(y_tr[mask], oof_s2_test[mask])
    print(f"  refine_strength={rs_val:.1f}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_rs = r, rs_val

print(f"\nBest refine_strength={best_rs:.1f}  OOF RAE={best_rae_v:.4f}")

# Recompute OOF with best refine_strength
oof = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    sim_va_te = sim_tr_te[va_idx]
    stage2_va, _, _ = reverse_delta_refine(
        fps_tr[va_idx], fps_te, phys_arr[va_idx], phys_te,
        sim_va_te, te_stage1, oof_stage1[va_idx], delta_model,
        refine_strength=best_rs
    )
    oof[va_idx] = stage2_va

m_best = full_metrics(y_tr, oof, cliff_pairs, f"best_rs={best_rs:.1f}")
print("\n" + pd.DataFrame([m_dir, m_s1, m_s2, m_best],
                           index=["direct","stage1","rs=1.0",f"rs={best_rs:.1f}"]).round(4).to_string())


Sweeping refine_strength...


  refine_strength=0.1  RAE=0.3269


  refine_strength=0.2  RAE=0.3272


  refine_strength=0.3  RAE=0.3275


  refine_strength=0.5  RAE=0.3281


  refine_strength=0.7  RAE=0.3288


  refine_strength=1.0  RAE=0.3297

Best refine_strength=0.1  OOF RAE=0.3269


  [best_rs=0.1] RAE=0.3269 MAE=0.2975 R2=0.8176 r=0.9059 rho=0.8753 tau=0.7236

           RAE     MAE      R2  Pearson  Spearman  Kendall
direct  0.5643  0.5134  0.5991   0.7740    0.7268   0.5345
stage1  0.3266  0.2972  0.8178   0.9060    0.8754   0.7237
rs=1.0  0.3297  0.3000  0.8155   0.9049    0.8737   0.7211
rs=0.1  0.3269  0.2975  0.8176   0.9059    0.8753   0.7236


In [10]:
# --- Final test predictions ---
# Stage 2 on test: for each test compound, find its nearest test neighbor
# and apply reverse delta using those predictions as anchors (self-refinement)
# This creates a test-test refinement loop: use sim^2-weighted average of
# stage1 predictions from test neighbors to refine each test compound

print("\nBuilding final test predictions...", flush=True)
# Test-test similarity
dot_tete = (fps_te @ fps_te.T).astype(np.float32)
rs_te2 = fps_te.sum(1).astype(np.float32)
union_tete = rs_te2[:,None] + rs_te2[None,:] - dot_tete
sim_te_te = np.where(union_tete>0, dot_tete/union_tete, 0.0)
np.fill_diagonal(sim_te_te, 0.0)

# Refine test predictions using test-test neighborhood
te_stage2 = np.copy(te_stage1)
for ti in range(len(te_stage1)):
    best_te_j = sim_te_te[ti].argmax()
    best_sim  = sim_te_te[ti, best_te_j]
    if best_sim < SIM_REFINE:
        continue
    fp_te_j = fps_te[best_te_j:best_te_j+1]
    fp_te_i = fps_te[ti:ti+1]
    sim_col = np.array([[best_sim]])
    te_pec50_j = np.array([te_stage1[best_te_j]])
    phys_d = phys_te[ti:ti+1] - phys_te[best_te_j:best_te_j+1]
    F_rev = make_delta_feats(fp_te_j, fp_te_i, sim_col, te_pec50_j, phys_d)
    delta_rev = delta_model.predict(F_rev)[0]
    te_via_nb = te_stage1[best_te_j] + delta_rev
    w = float(np.clip((best_sim - SIM_REFINE) / (1.0 - SIM_REFINE + 1e-6), 0.0, 1.0)) * best_rs
    w = float(np.clip(w, 0.0, 1.0))
    te_stage2[ti] = w * te_via_nb + (1.0 - w) * te_stage1[ti]

te_preds = np.clip(te_stage2, y_tr.min()-0.5, y_tr.max()+0.5)
print(f"Test stage2: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")

np.save(DATA_PROCESSED/"oof_reverse_delta_ml.npy", oof)
np.save(DATA_PROCESSED/"te_oof_reverse_delta_ml.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"106_reverse_delta_ml.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"\n*** nb106 OOF RAE = {m_best['RAE']:.4f} ***")


Building final test predictions...


Test stage2: min=2.79 med=4.72 max=5.73
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\106_reverse_delta_ml.csv

*** nb106 OOF RAE = 0.3269 ***
